# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 10
EPOCHS = 1
TOP_K = 1  # Number of top trials to save

TOTAL_NUM_PORTS = 100
n = 2

mixed_precision.set_global_policy("mixed_float16")

#? Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_dnn_v0" # None or "runs/nas_1" 

In [ ]:
RUN_DIR = RESUME_TRAINING_PATH or aa.create_run_directory(prefix="nas_")
print(f"Run directory: {RUN_DIR}")

## 3. Data Loading and Preprocessing

In [ ]:
# --------------------- Load the dataset in matlab format -------------------- #
dataset = scipy.io.loadmat(
    "/media/matheus/SSD/Projects/FluiDRA/src/data/w1_u1_n100//SNR_events_W1.0_U1_N100_kappa1.0e-16_mu1.0_m50.0.mat"
)["SNR_events"]

# Subsample data
dataset = dataset[: int(0.001 * dataset.shape[0]), :]

print(f"Shape of the data after configuration: {dataset.shape}\n")

## 4. Getters

### 4.1. Regularizers

In [ ]:
def get_regularizer(trial: optuna.Trial, name: str) -> Optional[tf.keras.regularizers.Regularizer]:
    """
    Suggests a regularization strategy using Optuna and returns the corresponding Keras regularizer.
    
    Args:
        trial (optuna.Trial): Optuna trial object used to sample the regularizer.
        name (str): Unique identifier for this regularizer parameter (used as key).

    Returns:
        Optional[tf.keras.regularizers.Regularizer]: The selected Keras regularizer instance,
        or `None` if "none" was selected.
    """
    # Suggest a regularizer type
    reg_type: str = trial.suggest_categorical(
        name,
        [
            "none",
            "l1",
            "l2",
            "l1l2",
            # "orthogonal",  #! only works for rank-2 tensors
        ],
    )

    # Map each regularizer name to a corresponding Keras regularizer instance
    regularizer_map: Dict[str, Optional[tf.keras.regularizers.Regularizer]] = {
        "none": None,
        "l1": regularizers.L1(l1=0.01),
        "l2": regularizers.L2(l2=0.01),
        "l1l2": regularizers.L1L2(l1=0.01, l2=0.01),
        "orthogonal": regularizers.OrthogonalRegularizer(factor=0.01, mode="rows"),
    }

    # Return the appropriate regularizer, or None if not found
    return regularizer_map.get(reg_type, None)

### 4.2. Activation Functions

In [ ]:
def get_activation(trial: Any, name: str) -> Union[str, Callable[..., layers.Layer]]:
    """
    Suggests an activation function from a predefined list using Optuna.

    Args:
        trial (Any): The Optuna trial instance used to suggest a value.
        name (str): A unique name for this hyperparameter (e.g., "layer_1_activation").

    Returns:
        Union[str, Callable[..., layers.Layer]]: A string representing the activation function.
        This can be passed directly into a Keras layer's `activation=` argument.
    """
    return trial.suggest_categorical(
        name,
        [
            "relu",
            "tanh",
            "sigmoid",  # Logistic
            "elu", 
            "swish",  # x * sigmoid(x)
            "leaky_relu",
        ],
    )

### 4.3. Optimizers

In [ ]:
def get_optimizer(trial: optuna.Trial) -> tf.keras.optimizers.Optimizer:
    """
    Suggests and returns a TensorFlow optimizer with a trial-based learning rate.

    Args:
        trial (optuna.Trial): Optuna trial object used for hyperparameter suggestion.

    Returns:
        tf.keras.optimizers.Optimizer: An instance of the selected optimizer.
    """
    # Suggest optimizer name from a predefined categorical set
    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "AdamW",
            "SGD",
            "Adam",
            "RMSprop",
            "Nadam",
            "Lion",
        ],
    )

    # Suggest learning rate on a logarithmic scale between 1e-5 and 1e-2
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Mapping of optimizer names to their TensorFlow classes
    optimizer_map: Dict[str, Type[tf.keras.optimizers.Optimizer]] = {
        "Adam": optimizers.Adam,
        "AdamW": optimizers.AdamW,
        "SGD": optimizers.SGD,
        "RMSprop": optimizers.RMSprop,
        "Nadam": optimizers.Nadam,
        "Lion": optimizers.Lion,
    }

    # Raise error if selected optimizer is not supported in the current context
    if optimizer_name not in optimizer_map:
        raise ValueError(
            f"Optimizer '{optimizer_name}' is not supported. "
            f"Supported optimizers are: {list(optimizer_map.keys())}."
        )

    # Instantiate and return the selected optimizer with suggested learning rate
    return optimizer_map[optimizer_name](learning_rate=learning_rate)

### 4.4. Callbacks

In [ ]:
def get_callbacks(trial: optuna.Trial, checkpoint_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        checkpoint_dir (str): Directory where model weights will be saved.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Construct path for saving weights for this specific trial
    checkpoint_path: str = os.path.join(checkpoint_dir, f"trial_{trial.number}.weights.h5")

    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )

    # Save only the best model weights based on monitored metric
    model_checkpoint = callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=monitor,
        save_best_only=True,  # only save weights if val_loss improves
        save_weights_only=True,  # save only the weights (not full model)
        verbose=0,
    )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = NanLossPrunerCallback(trial)

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, model_checkpoint, nan_pruner_callback, pruning_callback]

### 4.5. Scalers

In [ ]:
def get_scaler(
    trial: optuna.Trial,
) -> Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
    """
    Suggests and returns a scikit-learn scaler based on Optuna hyperparameter selection.

    Args:
        trial (optuna.Trial): Optuna trial object used to suggest hyperparameters.

    Returns:
        Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
            Instantiated scaler object from scikit-learn.
    """
    # Suggest a scaler name from the list of supported options
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",  # For normally-distributed data
            "MinMaxScaler_-1_1",  # Normalize to [-1, 1] range
            "MinMaxScaler_0_1",  # Normalize to [0, 1] range
            "RobustScaler",  # For data with outliers
            "QuantileTransformer",  # For non-normal or skewed data
            "PowerTransformer",  # For heavy-tailed or skewed data
        ],
    )

    # Return the appropriate scaler instance based on selection
    if scaler_name == "StandardScaler":
        return StandardScaler()
    elif scaler_name == "RobustScaler":
        return RobustScaler()
    elif scaler_name == "QuantileTransformer":
        return QuantileTransformer(output_distribution="normal")
    elif scaler_name == "PowerTransformer":
        return PowerTransformer(method="yeo-johnson")
    elif scaler_name == "MinMaxScaler_0_1":
        return MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        return MinMaxScaler(feature_range=(-1, 1))

    # Catch invalid or unknown choices
    else:
        raise ValueError(f"Unknown scaler selected: {scaler_name}")

### 4.6. Implementation getters

In [ ]:
def get_observed_ports(sinr_data, num_observed_ports, total_ports):
    """
    Extracts SINR values for the specified number of observed ports.

    The function selects a subset of SINR data by identifying equally spaced ports based on the
    number of observed ports specified. It returns the SINR values for these observed ports and
    their corresponding indices.

    Args:
        sinr_data (numpy.ndarray): A 2D array where each row represents an observation and each column
                                   represents a port with its corresponding SINR values.
        num_observed_ports (int): The number of observed ports to select from the SINR data.
        total_ports (int): The total number of ports in the SINR data.

    Returns:
        observed_sinr (numpy.ndarray): A 2D array containing the SINR values for the observed ports.
        observed_indices (numpy.ndarray): A 1D array of the indices corresponding to the observed ports.
    """
    observed_indices = np.linspace(0, total_ports - 1, num_observed_ports, dtype=int)
    observed_sinr = sinr_data[:, observed_indices]

    return observed_sinr, observed_indices


def getOP(
    observed_indices: np.ndarray, 
    predicted_values: np.ndarray, 
    true_values: np.ndarray, 
    threshold: float, 
    snr_linear: float,
    total_ports: int
) -> float:
    """Estimate the outage probability for regression models.

    This function compares the predicted and observed signal values at different 
    channels (ports) and determines whether the chosen signal is above a given threshold. 
    The outage probability is then computed as the proportion of times the signal falls 
    below this threshold.

    Args:
        observed_indices (np.ndarray): Indices of the observed ports (channels).
        predicted_values (np.ndarray): Matrix of predicted values for each sample.
        true_values (np.ndarray): Ground-truth values for each port.
        threshold (float): Threshold value for determining outage.
        snr_linear (float): Signal-to-noise ratio in linear scale.

    Returns:
        float: Estimated outage probability.
    """
    
    # Initialize an array with negative infinity to store the observed values
    observed_values_matrix = np.full((true_values.shape[0], total_ports), -np.inf, dtype=np.float64)

    # Assign the true values of the observed ports (channels) to the matrix
    observed_values_matrix[:, observed_indices] = true_values[:, observed_indices]

    # Find the index of the highest predicted value for each sample
    best_predicted_indices = np.argmax(predicted_values, axis=1)

    # Initialize an array with negative infinity to store the predicted values
    predicted_values_matrix = np.full((true_values.shape[0], total_ports), -np.inf, dtype=np.float64)

    # Assign the true value corresponding to the predicted best port
    predicted_values_matrix[np.arange(len(best_predicted_indices)), best_predicted_indices] = (
        true_values[np.arange(len(best_predicted_indices)), best_predicted_indices]
    )

    # Take the element-wise maximum between the observed and predicted value matrices
    best_value_matrix = np.maximum(observed_values_matrix, predicted_values_matrix)

    # print("Shape of Best Value Matrix:", best_value_matrix.shape)

    # Find the index of the best predicted or observed port (channel) for each sample
    best_predicted_or_observed_ports = np.argmax(best_value_matrix, axis=1)

    # print("Shape of Best Predicted/Observed Ports:", best_predicted_or_observed_ports.shape)
    # print("Number of Selected Ports:", len(best_predicted_or_observed_ports))

    # Retrieve the actual values corresponding to the best selected ports
    selected_values = best_value_matrix[np.arange(len(true_values)), best_predicted_or_observed_ports]

    # print("Shape of Selected Values:", selected_values.shape)

    # Determine which selected values are above the given threshold
    above_threshold = selected_values > (threshold / snr_linear)

    # print("Shape of Above Threshold Array:", above_threshold.shape)

    # Compute the outage probability: probability that the selected value is below the threshold
    outage_probability = 1.0 - (np.sum(above_threshold) / len(true_values))

    return outage_probability


## 5. Layers Builders

### 5.1. DNN

In [ ]:
def build_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_const",
) -> layers.Layer:
    """
    Build a deep neural network.

    Every hidden layer can have a different number of units. Supports optional
    batch normalization, dropout, weight/activity regularization, and
    two residual-connection strategies: "beside" (sequential) or "all"
    (dense skip connections).

    Args:
        trial (optuna.trial.Trial): Trial object for hyperparameter sampling.
        x (tf.keras.layers.Layer): Input tensor to the network.
        max_layers (int): Maximum number of Dense layers to include.
        max_units (int): Upper bound on units per layer.
        min_units (int): Lower bound on units per layer.
        step (int): Step size for sampling the number of units.
        try_batch_norm (bool): If True, sample whether to apply
            BatchNormalization after each Dense.
        use_regularization (bool): If True, sample kernel, bias, and activity
            regularizers per layer.
        residual_method (Optional[str]): Residual connection strategy:
            - "beside": Add a single running residual alongside layers.
            - "all": Add skip connections from all previous layers.
            - None: No residual connections.
        custom_name (str): Prefix for naming layers and hyperparameters.

    Returns:
        tf.keras.layers.Layer: Output tensor after applying all layers.
    """

    # Buffers for residual logic
    residual_buffer: Optional[layers.Layer] = None
    skip_buffers: list[layers.Layer] = []

    for i in range(max_layers):
        units = trial.suggest_int(f"{custom_name}_units_{i}", min_units, max_units, step=step)
        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        # Optional batch normalization
        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        # Dropout for regularization
        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # ——— Residual connection logic ———
        if residual_method == "beside":
            # Single running residual passed "beside" the main path
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    # Align dimensions if needed
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                    residual_buffer = x
                else:
                    residual_buffer = x

        elif residual_method == "all":
            # Dense skip connections from every previous layer
            if i == 0:
                skip_buffers = [x]
            else:
                to_add: list[layers.Layer] = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        adjusted = prev
                        # Align dimensions if needed
                        if adjusted.shape[-1] != x.shape[-1]:
                            adjusted = layers.Dense(
                                units=x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(adjusted)
                        to_add.append(adjusted)
                if to_add:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + to_add)
                skip_buffers.append(x)

    return x


def build_constant_width_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_const",
) -> layers.Layer:
    """
    Build a constant-width deep neural network.

    Every hidden layer has the same number of units. Supports optional
    batch normalization, dropout, weight/activity regularization, and
    two residual-connection strategies: "beside" (sequential) or "all"
    (dense skip connections).

    Args:
        trial (optuna.trial.Trial): Trial object for hyperparameter sampling.
        x (tf.keras.layers.Layer): Input tensor to the network.
        max_layers (int): Maximum number of Dense layers to include.
        max_units (int): Upper bound on units per layer.
        min_units (int): Lower bound on units per layer.
        step (int): Step size for sampling the number of units.
        try_batch_norm (bool): If True, sample whether to apply
            BatchNormalization after each Dense.
        use_regularization (bool): If True, sample kernel, bias, and activity
            regularizers per layer.
        residual_method (Optional[str]): Residual connection strategy:
            - "beside": Add a single running residual alongside layers.
            - "all": Add skip connections from all previous layers.
            - None: No residual connections.
        custom_name (str): Prefix for naming layers and hyperparameters.

    Returns:
        tf.keras.layers.Layer: Output tensor after applying all layers.
    """
    # Sample the number of layers and the constant width
    layers_count = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units = trial.suggest_int(f"{custom_name}_units", min_units, max_units, step=step)

    # Buffers for residual logic
    residual_buffer: Optional[layers.Layer] = None
    skip_buffers: list[layers.Layer] = []

    for i in range(layers_count):
        # Dense layer with sampled activation and optional regularizers
        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        # Optional batch normalization
        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        # Dropout for regularization
        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # ——— Residual connection logic ———
        if residual_method == "beside":
            # Single running residual passed "beside" the main path
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    # Align dimensions if needed
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                    residual_buffer = x
                else:
                    residual_buffer = x

        elif residual_method == "all":
            # Dense skip connections from every previous layer
            if i == 0:
                skip_buffers = [x]
            else:
                to_add: list[layers.Layer] = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        adjusted = prev
                        # Align dimensions if needed
                        if adjusted.shape[-1] != x.shape[-1]:
                            adjusted = layers.Dense(
                                units=x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(adjusted)
                        to_add.append(adjusted)
                if to_add:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + to_add)
                skip_buffers.append(x)

    return x


def build_inverted_funnel_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_inv_funnel",
) -> layers.Layer:
    """
    Build an inverted-funnel (bottleneck) deep neural network.

    Layer widths start narrow, increase linearly to a midpoint, then
    decrease symmetrically back to the edge width. Supports batch-norm,
    dropout, regularization, and the same residual options as constant.

    Args:
        trial (optuna.trial.Trial): Trial for hyperparameter sampling.
        x (tf.keras.layers.Layer): Input tensor.
        max_layers (int): Maximum number of layers.
        max_units (int): Maximum units allowed at the midpoint.
        min_units (int): Minimum units at the edges.
        step (int): Step size for sampling unit counts.
        try_batch_norm (bool): Whether to sample batch-normalization usage.
        use_regularization (bool): Whether to apply weight/activity regularizers.
        residual_method (Optional[str]): See `build_constant_width_dnn`.
        custom_name (str): Prefix for naming.

    Returns:
        tf.keras.layers.Layer: Output tensor after the inverted-funnel DNN.
    """
    layers_count = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_edge = trial.suggest_int(f"{custom_name}_units_edge", min_units, max_units, step=step)
    units_mid = trial.suggest_int(f"{custom_name}_units_mid", units_edge, max_units, step=step)

    # Calculate midpoint index (can be non-integer if even number of layers)
    mid_index = (layers_count - 1) / 2.0

    residual_buffer: Optional[layers.Layer] = None
    skip_buffers: list[layers.Layer] = []

    for i in range(layers_count):
        # Compute distance ratio from layer i to the center
        # (0 at center, 1 at edges), then interpolate width
        distance = abs(i - mid_index) / mid_index if mid_index != 0 else 0.0
        units = int(units_mid - (units_mid - units_edge) * distance)
        units = max(16, units)  # enforce a floor

        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # Residual logic identical to constant-width version
        if residual_method == "beside":
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                    residual_buffer = x
                else:
                    residual_buffer = x

        elif residual_method == "all":
            if i == 0:
                skip_buffers = [x]
            else:
                residuals: list[layers.Layer] = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        adjusted = prev
                        if adjusted.shape[-1] != x.shape[-1]:
                            adjusted = layers.Dense(
                                units=x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(adjusted)
                        residuals.append(adjusted)
                if residuals:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals)
                skip_buffers.append(x)

    return x


def build_hourglass_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_hourglass",
) -> layers.Layer:
    """
    Build an hourglass-shaped DNN: narrow→wide→plateau→wide→narrow.

    The network ramps up from a small width at the edges to a maximum
    width, holds that width for a sample-defined 'plateau' number of
    layers, then mirrors the ramp down.

    Args:
        trial (optuna.trial.Trial): Trial for hyperparameter sampling.
        x (tf.keras.layers.Layer): Input tensor.
        max_layers (int): Maximum total number of layers.
        max_units (int): Maximum units at the plateau.
        min_units (int): Minimum units at the first/last layer.
        step (int): Step size for unit sampling.
        try_batch_norm (bool): Whether to sample batch-normalization.
        use_regularization (bool): Whether to apply regularizers.
        residual_method (Optional[str]): See other builders.
        custom_name (str): Naming prefix.

    Returns:
        tf.keras.layers.Layer: Output tensor after the hourglass DNN.
    """
    layers_count = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_edge = trial.suggest_int(f"{custom_name}_units_edge", min_units, max_units, step=step)
    units_mid = trial.suggest_int(f"{custom_name}_units_mid", units_edge, max_units, step=step)
    plateau = trial.suggest_int(f"{custom_name}_plateau", 1, layers_count)

    # Determine how many layers pre- and post-plateau
    pre = (layers_count - plateau) // 2
    post = layers_count - plateau - pre

    residual_buffer: Optional[layers.Layer] = None
    skip_buffers: list[layers.Layer] = []

    for i in range(layers_count):
        if i < pre:
            # Ramp up phase
            factor = i / pre if pre > 0 else 1.0
            units = int(units_edge + (units_mid - units_edge) * factor)
        elif i < pre + plateau:
            # Plateau at max width
            units = units_mid
        else:
            # Ramp down phase
            down_idx = i - (pre + plateau)
            factor = down_idx / post if post > 0 else 1.0
            units = int(units_mid - (units_mid - units_edge) * factor)

        units = max(16, units)  # ensure a minimum size

        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # Residual logic reused verbatim from other builders
        if residual_method == "beside":
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                    residual_buffer = x
                else:
                    residual_buffer = x

        elif residual_method == "all":
            if i == 0:
                skip_buffers = [x]
            else:
                residuals: list[layers.Layer] = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        adjusted = prev
                        if adjusted.shape[-1] != x.shape[-1]:
                            adjusted = layers.Dense(
                                units=x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(adjusted)
                        residuals.append(adjusted)
                if residuals:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals)
                skip_buffers.append(x)

    return x


def build_funnel_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    max_decay_factor: float = 0.9,
    min_decay_factor: float = 0.1,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn",
) -> layers.Layer:
    """
    Builds the DNN funnel, wide to narrow.

    Args:
        trial (Any): The optimization trial containing hyperparameters.
        x (layers.Layer): The input tensor
        max_layers (int): Maximum number of layers in the DNN.
        max_units (int): Maximum number of units in a layer.
        min_units (int): Minimum number of units in a layer.
        step (int): Step size for the number of units.
        max_decay_factor (float): Maximum decay factor for reducing units in subsequent layers.
        min_decay_factor (float): Minimum decay factor for reducing units in subsequent layers.
        try_batch_norm (bool): Whether to allow try batch normalization.
        use_regularization (bool): Whether to apply regularization on the layers.
        residual_method (Optional[str]): Type of residual connection method to use.
            - "beside": Adds residual connections between consecutive layers.
            - "all": Adds residual connections from all previous layers.
            - None: No residual connections are applied.
        custom_name (str): Custom prefix for layer names.

    Returns:
        layers.Layer: The processed output tensor.
    """
    dnn_layers = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_layer_0 = trial.suggest_int(f"{custom_name}_units_layer_0", min_units, max_units, step=step)
    decay_factor = trial.suggest_float(
        f"{custom_name}_decay_factor", min_decay_factor, max_decay_factor, step=0.1
    )

    residual_dense = None
    skip_connections_dense = []

    for i in range(dnn_layers):
        units = units_layer_0 if i == 0 else max(16, int(units_layer_0 * (decay_factor**i)))

        activation = get_activation(trial, f"{custom_name}_activation_layer_{i}")

        dnn_kernel_regularizer = (
            get_regularizer(trial, f"dnn_kernel_regularizer_layer_{i}") if use_regularization else None
        )
        dnn_bias_regularizer = (
            get_regularizer(trial, f"dnn_bias_regularizer_layer_{i}") if use_regularization else None
        )
        dnn_activity_regularizer = (
            get_regularizer(trial, f"dnn_activity_regularizer_layer_{i}") if use_regularization else None
        )
        x = layers.Dense(
            units=units,
            activation=activation,
            name=f"{custom_name}_dense_{i}",
            kernel_regularizer=dnn_kernel_regularizer,
            bias_regularizer=dnn_bias_regularizer,
            activity_regularizer=dnn_activity_regularizer,
        )(x)

        if try_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_layer_{i}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{i}")(x)

        dropout_rate = trial.suggest_float(f"{custom_name}_dropout_layer_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(dropout_rate, name=f"{custom_name}_dropout_{i}")(x)

        # ————————————————————————— Residual Connection Logic ———————————————————————— #
        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        #! EXPERIMENTAL FEATURE
        #! UNTESTED, UNSURE IF IT WORKS
        if residual_method == "beside":
            if i == 0:
                residual_dense = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}", [True, False]):
                    if residual_dense.shape[-1] != x.shape[-1]:
                        residual_dense = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_residual_dense_{i}"
                        )(residual_dense)
                    x = layers.Add(name=f"{custom_name}_residual_add_{i}")([x, residual_dense])
                    residual_dense = x
                else:
                    residual_dense = x

        elif residual_method == "all":
            if i == 0:
                skip_connections_dense = [x]
            else:
                residuals_to_add = []
                for j, prev in enumerate(skip_connections_dense):
                    if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}_{j}", [True, False]):
                        adjusted_prev = prev

                        if adjusted_prev.shape[-1] != x.shape[-1]:
                            adjusted_prev = layers.Dense(
                                units=x.shape[-1],
                                activation=None,
                                name=f"{custom_name}_skip_residual_dense_{i}_{j}",
                            )(adjusted_prev)

                        residuals_to_add.append(adjusted_prev)
                if residuals_to_add:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals_to_add)
                skip_connections_dense.append(x)

    return x

### 5.2. CNN

In [ ]:
def build_cnn1d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 10,
    min_pool_size: int = 2,
    max_pool_size: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn1d",
) -> layers.Layer:
    """
    Builds a 1D CNN where Optuna picks filters, kernel sizes, and pooling window per layer.

    Args:
        trial (optuna.Trial): Optuna trial object.
        x (Layer): Input Keras tensor.
        num_layers (int): Number of Conv1D + Pool1D blocks.
        max_filters (int): Upper bound on number of filters.
        min_filters (int): Lower bound on number of filters.
        filter_step (int): Step size when sampling number of filters.
        max_kernel_size (int): Maximum size of the 1D convolution kernel.
        min_pool_size (int): Minimum size of the 1D pooling window.
        max_pool_size (int): Maximum size of the 1D pooling window.
        use_batch_norm (bool): If True, trial may enable BatchNormalization per layer.
        use_regularization (bool): If True, trial picks kernel, bias, and activity regularizers.
        residual_method (Optional[str]): One of {None, "beside", "all"} to control skip connections.
        custom_name (str): Prefix for naming all layers.

    Returns:
        Layer: Output tensor after applying all CNN blocks.
    """

    # Placeholders for residual connection strategies
    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # 1) Sample pooling window size
        pool_size = trial.suggest_int(
            f"{custom_name}_pool_size_{layer_idx}", min_pool_size, max_pool_size
        )

        # 2) Sample number of filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 3) Sample convolution kernel size
        kernel_size = trial.suggest_int(
            f"{custom_name}_kernel_size_{layer_idx}", 1, max_kernel_size
        )

        # 4) Activation function
        activation_fn = get_activation(trial, f"{custom_name}_activation_{layer_idx}")

        # 5) Regularizers (if enabled)
        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_{layer_idx}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_{layer_idx}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_{layer_idx}")
            if use_regularization
            else None
        )

        # 6) 1D convolution
        x = layers.Conv1D(
            filters=num_filters,
            kernel_size=kernel_size,
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_conv1d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # 7) Optional BatchNormalization
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{layer_idx}")(x)

        # 8) Residual connections
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(
                    f"{custom_name}_use_residual_{layer_idx}", [True, False]
                ):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    # Align channel dimensions if needed
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv1D(
                            filters=target_ch,
                            kernel_size=1,
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(name=f"{custom_name}_res_add_{layer_idx}")([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_{layer_idx}_{prev_idx}", [True, False]
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv1D(
                                filters=target_ch,
                                kernel_size=1,
                                padding="same",
                                name=f"{custom_name}_skip_res_conv1d_{layer_idx}_{prev_idx}",
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(
                        name=f"{custom_name}_res_all_add_{layer_idx}"
                    )([x] + to_add)
                all_skip_connections.append(x)

        # 9) 1D MaxPooling
        x = layers.MaxPooling1D(
            pool_size=pool_size, name=f"{custom_name}_maxpool_{layer_idx}"
        )(x)

    return x


def build_cnn2d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 10,
    min_pool_size_dim1: int = 2,
    max_pool_size_dim1: int = 2,
    min_pool_size_dim2: int = 2,
    max_pool_size_dim2: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn",
) -> layers.Layer:
    """
    Builds a 2D CNN where Optuna picks filters, kernels, and pooling window per layer.

    Args:
        trial: Optuna trial object.
        x: Input Keras tensor.
        num_layers: Number of Conv2D+Pool blocks.
        max_filters: Upper bound on filters.
        min_filters: Lower bound on filters.
        filter_step: Step size for filters.
        max_kernel_size: Max kernel dim for height & width.
        min_pool_size_dim1: Min pooling window height.
        max_pool_size_dim1: Max pooling window height.
        min_pool_size_dim2: Min pooling window width.
        max_pool_size_dim2: Max pooling window width.
        use_batch_norm: If True, trial.prunes BatchNorm on/off.
        use_regularization: If True, trial selects kernel/bias/activity regularizers.
        residual_method: One of {None, "beside", "all"}.
        custom_name: Prefix for naming each layer.

    Returns:
        The output tensor after all blocks.
    """

    # Containers for residual strategies
    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # 0) Sample a single pooling window to use in every block
        pool_dim1 = trial.suggest_int(
            f"{custom_name}_pool_size_dim1_{layer_idx}",
            min_pool_size_dim1,
            max_pool_size_dim1,
        )
        pool_dim2 = trial.suggest_int(
            f"{custom_name}_pool_size_dim2_{layer_idx}",
            min_pool_size_dim2,
            max_pool_size_dim2,
        )
        pool_size: Tuple[int, int] = (pool_dim1, pool_dim2)
        
        # 1) Filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_layer_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 2) Kernel dims
        kernel_h = trial.suggest_int(f"{custom_name}_kernel_height_{layer_idx}", 1, max_kernel_size)
        kernel_w = trial.suggest_int(f"{custom_name}_kernel_width_{layer_idx}", 1, max_kernel_size)

        # Activation
        activation_fn = get_activation(trial, f"{custom_name}_activation_layer_{layer_idx}")

        # Regularizers
        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )

        # Conv2D
        x = layers.Conv2D(
            filters=num_filters,
            kernel_size=(kernel_h, kernel_w),
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_conv2d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # 3) Optional BatchNorm
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_layer_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{layer_idx}")(x)

        # 4) Residuals
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{layer_idx}", [True, False]):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv2D(
                            filters=target_ch,
                            kernel_size=(1, 1),
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(name=f"{custom_name}_res_add_{layer_idx}")([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_layer_{layer_idx}_{prev_idx}",
                        [True, False],
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv2D(
                                filters=target_ch,
                                kernel_size=(1, 1),
                                padding="same",
                                name=(f"{custom_name}_skip_res_conv2d_" f"{layer_idx}_{prev_idx}"),
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(name=f"{custom_name}_res_all_add_{layer_idx}")([x] + to_add)
                all_skip_connections.append(x)

        # 5) MaxPooling with sampled window
        x = layers.MaxPooling2D(pool_size=pool_size, name=f"{custom_name}_maxpool_{layer_idx}")(x)

    return x


def build_cnn3d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 5,
    min_pool_size_dim1: int = 2,
    max_pool_size_dim1: int = 2,
    min_pool_size_dim2: int = 2,
    max_pool_size_dim2: int = 2,
    min_pool_size_dim3: int = 2,
    max_pool_size_dim3: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn3d",
) -> layers.Layer:
    """
    Builds a 3D CNN where Optuna picks filters, kernels, and pooling window per layer.

    Args:
        trial: Optuna Trial for hyperparameter sampling.
        x: Input Keras 5D tensor (batch, D, H, W, C).
        num_layers: Number of Conv3D+Pool blocks.
        max_filters: Upper bound on the number of filters per layer.
        min_filters: Lower bound on the number of filters per layer.
        filter_step: Step size for filter suggestions.
        max_kernel_size: Max dimension for kernel depth/height/width.
        min_pool_size_dim1: Min depth of pooling window.
        max_pool_size_dim1: Max depth of pooling window.
        min_pool_size_dim2: Min height of pooling window.
        max_pool_size_dim2: Max height of pooling window.
        min_pool_size_dim3: Min width of pooling window.
        max_pool_size_dim3: Max width of pooling window.
        use_batch_norm: If True, trial decides per-layer BatchNormalization.
        use_regularization: If True, trial samples kernel/bias/activity regularizers.
        residual_method: One of {None, "beside", "all"} residual strategies.
        custom_name: Prefix for naming layers for traceability.

    Returns:
        The output tensor after all Conv3D, normalization, residual, and pooling layers.
    """

    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # Sample pooling window
        pool_d = trial.suggest_int(
            f"{custom_name}_pool_size_dim1_{layer_idx}",
            min_pool_size_dim1,
            max_pool_size_dim1,
        )
        pool_h = trial.suggest_int(
            f"{custom_name}_pool_size_dim2_{layer_idx}",
            min_pool_size_dim2,
            max_pool_size_dim2,
        )
        pool_w = trial.suggest_int(
            f"{custom_name}_pool_size_dim3_{layer_idx}",
            min_pool_size_dim3,
            max_pool_size_dim3,
        )
        pool_size: Tuple[int, int, int] = (pool_d, pool_h, pool_w)
        
        
        # 1) Number of filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_layer_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 2) Kernel dimensions
        kernel_d = trial.suggest_int(
            f"{custom_name}_kernel_depth_{layer_idx}", 1, max_kernel_size
        )
        kernel_h = trial.suggest_int(
            f"{custom_name}_kernel_height_{layer_idx}", 1, max_kernel_size
        )
        kernel_w = trial.suggest_int(
            f"{custom_name}_kernel_width_{layer_idx}", 1, max_kernel_size
        )
        kernel_size = (kernel_d, kernel_h, kernel_w)

        # Activation
        activation_fn = get_activation(
            trial, f"{custom_name}_activation_layer_{layer_idx}"
        )

        # Regularizers
        kernel_reg = (
            get_regularizer(
                trial, f"{custom_name}_kernel_regularizer_layer_{layer_idx}"
            )
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(
                trial, f"{custom_name}_bias_regularizer_layer_{layer_idx}"
            )
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(
                trial, f"{custom_name}_activity_regularizer_layer_{layer_idx}"
            )
            if use_regularization
            else None
        )

        # Conv3D
        x = layers.Conv3D(
            filters=num_filters,
            kernel_size=kernel_size,
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_conv3d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # Optional BatchNorm
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_layer_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(
                name=f"{custom_name}_batch_norm_{layer_idx}"
            )(x)

        # Residual connections
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(
                    f"{custom_name}_use_residual_layer_{layer_idx}", [True, False]
                ):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv3D(
                            filters=target_ch,
                            kernel_size=(1, 1, 1),
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(
                        name=f"{custom_name}_res_add_{layer_idx}"
                    )([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_layer_{layer_idx}_{prev_idx}",
                        [True, False],
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv3D(
                                filters=target_ch,
                                kernel_size=(1, 1, 1),
                                padding="same",
                                name=(
                                    f"{custom_name}_skip_res_conv3d_"
                                    f"{layer_idx}_{prev_idx}"
                                ),
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(
                        name=f"{custom_name}_res_all_add_{layer_idx}"
                    )([x] + to_add)
                all_skip_connections.append(x)

        # MaxPooling3D with sampled window
        x = layers.MaxPooling3D(
            pool_size=pool_size,
            name=f"{custom_name}_maxpool3d_{layer_idx}"
        )(x)

    return x

### 5.3. LSTM


In [ ]:
def build_lstm(
    trial: optuna.Trial,
    x: layers.Layer,
    max_layers: int = 5,
    max_units: int = 256,
    min_units: int = 32,
    unit_step: int = 32,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "lstm",
) -> layers.Layer:
    """
    Build a stacked 1D LSTM with optional BatchNormalization and residual connections.

    Args:
        trial (optuna.Trial): Trial for hyperparameter suggestions.
        x (Layer): Input tensor.
        max_layers (int): Maximum number of LSTM blocks. Defaults to 5.
        max_units (int): Maximum number of units for the first block. Defaults to 256.
        min_units (int): Minimum number of units for the first block. Defaults to 32.
        unit_step (int): Step size when sampling the number of units. Defaults to 32.
        try_batch_norm (bool): Whether to allow BatchNormalization after each LSTM. Defaults to False.
        use_regularization (bool): Whether to apply kernel/bias/activity regularizers. Defaults to False.
        residual_method (Optional[str]): One of {"beside", "all", None} for skip connections. Defaults to None.
        custom_name (str): Prefix for naming layers. Defaults to "lstm".

    Returns:
        Layer: Tensor after successive LSTM → (BatchNorm → Residual) blocks.
    """
    residual_tensor = None
    skip_tensors = []

    for i in range(max_layers):
        units = trial.suggest_int(
            f"{custom_name}_units_layer_{i}", min_units, max_units, step=unit_step
        )

        activation = get_activation(trial, f"{custom_name}_activation_layer_{i}")

        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_{i}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_{i}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_{i}")
            if use_regularization
            else None
        )

        x = layers.LSTM(
            units=units,
            activation=activation,
            return_sequences=(i < max_layers - 1),
            recurrent_dropout=trial.suggest_float(
                f"lstm_recurrent_dropout_layer_{i}", 0.0, 0.5, step=0.1
            ),
            dropout=trial.suggest_float(
                f"lstm_dropout_layer_{i}", 0.0, 0.5, step=0.1
            ),
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_lstm_{i}",
        )(x)

        if try_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_{i}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{i}")(x)

        # ————————————————————————— Residual Connection Logic ———————————————————————— #
        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        #! EXPERIMENTAL FEATURE
        #! UNTESTED, UNSURE IF IT WORKS
        if residual_method == "beside":
            if i == 0:
                residual_tensor = x
            else:
                if trial.suggest_categorical(
                    f"{custom_name}_use_residual_{i}", [True, False]
                ):
                    if residual_tensor.shape[-1] != x.shape[-1]:
                        residual_tensor = layers.Dense(
                            x.shape[-1],
                            activation=None,
                            name=f"{custom_name}_res_dense_{i}",
                        )(residual_tensor)
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x, residual_tensor])
                    residual_tensor = x
                else:
                    residual_tensor = x

        elif residual_method == "all":
            if i == 0:
                skip_tensors = [x]
            else:
                additions = []
                for j, prev in enumerate(skip_tensors):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_skip_{i}_{j}", [True, False]
                    ):
                        adj = prev
                        if adj.shape[-1] != x.shape[-1]:
                            adj = layers.Dense(
                                x.shape[-1],
                                activation=None,
                                name=f"{custom_name}_skip_dense_{i}_{j}",
                            )(adj)
                        additions.append(adj)
                if additions:
                    x = layers.Add(name=f"{custom_name}_add_all_{i}")([x] + additions)
                skip_tensors.append(x)

    return x


### 5.2. TCNN

In [ ]:
def build_tcnn1d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 10,
    min_upsample_size: int = 2,
    max_upsample_size: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "tcnn1d",
) -> layers.Layer:
    """
    Builds a 1D TCNN where Optuna picks filters, kernel sizes, and upsampling window per layer.

    Args:
        trial (optuna.Trial): Optuna trial object.
        x (Layer): Input Keras tensor.
        num_layers (int): Number of Conv1D + Pool1D blocks.
        max_filters (int): Upper bound on number of filters.
        min_filters (int): Lower bound on number of filters.
        filter_step (int): Step size when sampling number of filters.
        max_kernel_size (int): Maximum size of the 1D convolution kernel.
        min_upsample_size (int): Minimum size of the 1D upsampling window.
        max_upsample_size (int): Maximum size of the 1D upsampling window.
        use_batch_norm (bool): If True, trial may enable BatchNormalization per layer.
        use_regularization (bool): If True, trial picks kernel, bias, and activity regularizers.
        residual_method (Optional[str]): One of {None, "beside", "all"} to control skip connections.
        custom_name (str): Prefix for naming all layers.

    Returns:
        Tensor after successive Conv1DTranspose → Activation → (BatchNorm → Residual) → UpSampling1D blocks.
    """

    # Placeholders for residual connection strategies
    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # 1) Sample pooling window size
        upsampling_window_size = trial.suggest_int(
            f"{custom_name}_pool_size_{layer_idx}", min_upsample_size, max_upsample_size
        )

        # 2) Sample number of filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 3) Sample convolution kernel size
        kernel_size = trial.suggest_int(
            f"{custom_name}_kernel_size_{layer_idx}", 1, max_kernel_size
        )

        # 4) Activation function
        activation_fn = get_activation(trial, f"{custom_name}_activation_{layer_idx}")

        # 5) Regularizers (if enabled)
        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_{layer_idx}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_{layer_idx}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_{layer_idx}")
            if use_regularization
            else None
        )

        # 6) 1D convolution
        x = layers.Conv1DTranspose(
            filters=num_filters,
            kernel_size=kernel_size,
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_tcnn1d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # 7) Optional BatchNormalization
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{layer_idx}")(x)

        # 8) Residual connections
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(
                    f"{custom_name}_use_residual_{layer_idx}", [True, False]
                ):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    # Align channel dimensions if needed
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv1DTranspose(
                            filters=target_ch,
                            kernel_size=1,
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(name=f"{custom_name}_res_add_{layer_idx}")([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_{layer_idx}_{prev_idx}", [True, False]
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv1DTranspose(
                                filters=target_ch,
                                kernel_size=1,
                                padding="same",
                                name=f"{custom_name}_skip_res_conv1d_{layer_idx}_{prev_idx}",
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(
                        name=f"{custom_name}_res_all_add_{layer_idx}"
                    )([x] + to_add)
                all_skip_connections.append(x)

        # 9) 1D UpSampling
        x = layers.UpSampling1D(
            size=upsampling_window_size, name=f"{custom_name}_upsample_{layer_idx}"
        )(x)

    return x

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    X: List[np.ndarray],
    y: List[np.ndarray],
    checkpoint_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        checkpoint_dir (str): Path to store checkpoint files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        use_regularization (bool): If True, adds regularization (e.g., L1/L2) to layers to prevent overfitting.
        residual_method (Optional[str]): tyoe of residual connection to use:
            - "beside": Adds residual connections between consecutive layers.
            - "all": test residual connections between all layers.
            - None: No residual connections are applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    # Each trial gets a different seed to split the data
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ————————————————————————————— Prepare the Data ————————————————————————————— #
    observed_ports = X[0]
    dataset = y[0]

    X_train, X_val, y_train, y_val = train_test_split(
        observed_ports,
        dataset,
        test_size=0.2,
        random_state=trial.number,
        shuffle=True,
    )

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # —————————————————————————————————— Scaler —————————————————————————————————— #
        scaler = get_scaler(trial)
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        # ——————————————————————————— Observed ports input ——————————————————————————— #
        inputs = layers.Input(shape=(observed_ports.shape[1],))  # Observed ports as input

        max_layers = trial.suggest_int("num_layers", 1, 6)

        x = build_dnn(
            trial=trial,
            x=inputs,
            max_layers=max_layers,
            max_units=2048,
            min_units=32,
            step=32,
            try_batch_norm=False,
            use_regularization=use_regularization,
            residual_method=residual_method,
            custom_name="dnn",
        )

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(TOTAL_NUM_PORTS, activation="linear")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=inputs, outputs=(outputs,))

        # ———————————————————————————— Vizualize the Model ——————————————————————————— #
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = get_optimizer(trial)
        model.compile(
            optimizer=optimizer,
            loss="mse",
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, checkpoint_dir),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = aa.compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = aa.compute_params_penalized_loss(loss=loss, model=model)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                 Trial Results                                #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
    
        # Create figure
        fig, ax_loss = plt.subplots(figsize=(8, 6))

        # Plot Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ————————————————————————————————— Evaluate ————————————————————————————————— #
        datasets = [dataset_1, dataset_2, dataset_3, dataset_4, dataset_5]
        test_losses = []

        n_ports = observed_ports.shape[1]

        for i, dataset in enumerate(datasets, start=1):
            observed_ports, _ = get_observed_ports(
                dataset, num_observed_ports=n_ports, total_ports=TOTAL_NUM_PORTS
            )
            
            # Separate X_train, y_train, X_test, y_test
            _, X_test, _, y_test = train_test_split(
                observed_ports,
                dataset,
                test_size=0.2,
                random_state=5555,
                shuffle=True,
            )
            
            X_test = scaler.transform(X_test)
            
            test_loss = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=0)
            test_losses.append(test_loss)

        # Unpack test losses for clarity
        dataset_test_losses = {f"dataset_{i}_test_loss": loss for i, loss in enumerate(test_losses, start=1)}

        # Print test losses in a formatted manner
        print("\n" + "=" * 15)
        for dataset_name, test_loss in dataset_test_losses.items():
            print(f"{dataset_name.replace('_', ' ').capitalize()}: {test_loss:.12f}")
        
        # write the dataset test losses to a text file
        test_losses_filepath = os.path.join(
            model_dir, f"trial_{trial.number}_test_losses.txt"
        )
        # we don't have any model hyperparams to dump here, so pass an empty dict
        aa.save_trial_params_to_file(
            filepath=test_losses_filepath,
            params={},
            **dataset_test_losses
        )

        params = model.count_params()
        print(f"\nNumber of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        
        trial.set_user_attr("num_params", params)
        trial.set_user_attr("model_size_mb", params * 4 / (1024 ** 2))

        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"❌ Trial {trial.number} hit OOM (ResourceExhaustedError): {oom_err}")

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(oom_err) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    except Exception as e:
        print(f"An error occurred during the trial execution: {e}")
        traceback.print_exc()

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(e) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [ ]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# aa.log_resources(log_dir=resources_dir)

In [ ]:
try:
    pid = os.getpid()
    cmd = (
        f'python3 "{os.path.abspath("_monitor_kernel_life.py")}" '
        f"--pid {pid} --custom-title {RUN_DIR}; exec bash"
    )
    terminals = [
        ["xfce4-terminal", "--disable-server", "--hold", "-e", f'bash -c "{cmd}"'],
        ["gnome-terminal", "--disable-factory", "--", "bash", "-i", "-c", cmd],
        ["xterm", "-hold", "-e", cmd],
        ["konsole", "--hold", "-e", f'bash -c "{cmd}"'],
    ]
    term = next((t for t in terminals if shutil.which(t[0])), None)
    if not term:
        raise RuntimeError(
            "No supported terminal emulator found; install gnome-terminal, "
            "xfce4-terminal, konsole, or xterm."
        )
    _monitor_proc = subprocess.Popen(term, preexec_fn=os.setpgrp)
    print(f"[INFO] Launched monitor in {term[0]} (PID={pid})")
except Exception as e:
    print(f"[ERROR] Auto launching kernel monitoring failed! {e}\n")
    display(
        HTML(
            f"Call the monitor script manually: "
            f'<span style="color: orange;">'
            f"python _monitor_kernel_life.py --pid {pid} --custom-title {RUN_DIR}"
            f"</span>"
        )
    )
    pass

## Main

In [ ]:

try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, f"optuna_study_{n}_ports")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "weights": os.path.join(study_dir, "weights"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    checkpoint_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["weights"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    print(f"Initializing study at '{study_dir}'...")
    
    # ——————————————————————————————————— Data ——————————————————————————————————— #
    observed_ports, observed_indices = get_observed_ports(
        dataset, num_observed_ports=n, total_ports=TOTAL_NUM_PORTS
    )
        
    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            X=[observed_ports],
            y=[dataset],
            checkpoint_dir=checkpoint_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            use_regularization=False,
            residual_method=None,  #! Find your backbone first
            show_summary=False,
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    valid_trials = [
        t for t in study.trials if t.value is not None and not (math.isnan(t.value) or math.isinf(t.value))
    ]
    sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_num_params = trial.user_attrs.get("num_params", None)
        trial_model_size = trial.user_attrs.get("model_size_mb", None)

        aa.save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            num_params=trial_num_params,
            model_size_mb=trial_model_size,
            sampler=study.sampler.__class__.__name__,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    cleanup_paths = [
        (checkpoint_dir, "trial_{trial_id}.weights.h5"),
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (model_dir, "trial_{trial_id}_test_losses.txt"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    aa.analyze_study(study, fig_dir=fig_dir, table_dir=study_dir)

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(
        1
        for t in study.trials
        if t.state not in {optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED}
    )
    notify_training_success(
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject=f"🎉 {RUN_DIR} with {n} ports Complete - Failed Trials: {failed_trials}",
        )
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

In [ ]:
# Kill the monitor kernel life process
if _monitor_proc is not None and _monitor_proc.poll() is None:
    os.killpg(_monitor_proc.pid, signal.SIGINT)